# Multi-DOF free vibration of undamped system
1. 关于 $M, C, K$ 的维度：该代码使用了 len(M) 动态获取维度。无论考试题目是 2-DOF、3-DOF 还是更多，你只需要调整 M, C, K 的矩阵大小（例如 $3 \times 3$ 或 $4 \times 4$），代码会自动适配求解。
2. 固有频率 $\omega_n$ 与特征值：代码中的 eig(inv(M) @ K) 对应课件中的公式 $(5.33)$ $\Delta(\omega^2) = \text{det}[K - \omega^2 M] = 0$。特征值的平方根即为 $\omega_n$（对应公式 $5.35$）。
3. 主振型（Mode Shape）：代码输出的 modes_undamped 每一列对应一个 $\mathbf{u}_i$（特征向量），这对应课件公式 $(5.39)$。这些向量决定了系统在特定频率下各质量块的相对位移比例（即振幅比 $u_{2i}/u_{1i}$）。
4. 阻尼项的处理：如果题目要求“保留阻尼”，通常是为了考察有阻尼频率 $\omega_d = \omega_n \sqrt{1-\zeta^2}$。代码中的状态空间法会自动计算 $\zeta$ 和 $\omega_d$。如果题目说“无阻尼”，你只需将矩阵 C 设置为 np.zeros((n, n)) 即可。

In [11]:
import numpy as np
from numpy.linalg import eig, inv

def solve_vibration_exam(M, C, K, F_vector, target_freq_hz=None):
    """
    M, C, K: 质量、阻尼、刚度矩阵
    F_vector: 激励力向量 (例如 [0, 0, 1] 表示力作用在第3个质量块上)
    target_freq_hz: 想要计算响应的具体频率。如果为 None，则默认使用最低固有频率
    """
    n = len(M)
    
    # 1. 固有频率与质量归一化振型 (u.T @ M @ u = 1)
    eigenvalues, eigenvectors = eig(inv(M) @ K)
    idx = np.argsort(eigenvalues)
    wn = np.sqrt(eigenvalues[idx])
    
    # 质量归一化
    u_norm = np.zeros_like(eigenvectors)
    for i in range(n):
        vi = eigenvectors[:, idx[i]]
        m_modal = vi.T @ M @ vi
        u_norm[:, i] = vi / np.sqrt(m_modal)

    # 2. 确定计算频率
    if target_freq_hz is None:
        f_calc = wn[0] / (2 * np.pi) # 默认指向最低固有频率
    else:
        f_calc = target_freq_hz
    
    w_calc = 2 * np.pi * f_calc

    # 3. 计算稳态响应位移 (Complex Response)
    # 阻抗矩阵 Z(iw) = -w^2*M + iw*C + K 
    Z = -w_calc**2 * M + 1j * w_calc * C + K
    G = inv(Z) # 传递函数矩阵 
    X = G @ F_vector # 位移响应向量 (复数形式)

    # 4. 计算峰峰值 (Peak-to-Peak)
    # 振幅 Amplitude = abs(X)，峰峰值 = 2 * Amplitude
    peak_to_peak = 2 * np.abs(X)

    # --- 输出结果 ---
    print("="*60)
    print(f"1. 固有频率 (rad/s): {wn}")
    print(f"2. 质量归一化振型矩阵 (每列为一个 Mode❗️):\n{u_norm}")
    print("-" * 60)
    print(f"3. 受迫振动分析 (频率: {f_calc:.4f} Hz)")
    print(f"   激励力向量 F: {F_vector} N")
    for i in range(n):
        print(f"   质量块 m{i+1} 的位移振幅: {np.abs(X[i]):.6e} m")
        print(f"   质量块 m{i+1} 的峰峰值 (P-P): {peak_to_peak[i]:.6e} m")
    print("="*60)


In [13]:
# ==========================================
# 考试修改区：在这里输入你的推导结果
# ==========================================

# 以一个典型的三自由度（3-DOF）系统为例：三个质量块 m, 之间用 k 和 c 连接
m1, m2, m3 = 0.5, 0.7, 0.3
k1, k2, k3 = 200, 300, 170 
c1, c2, c3 = 0.1, 0.2, 0.08  # 阻尼项

# 1. 质量矩阵 M
M = np.array([
    [m1, 0,  0],
    [0,  m2, 0],
    [0,  0,  m3]
])

# 2. 刚度矩阵 K (根据你的 EOM 推导填写)
K = np.array([
    [k1+k2, -k2,    0],
    [-k2,    k2+k3, -k3],
    [0,     -k3,    k3]
])

# 3. 阻尼矩阵 C (如果没有阻尼，设为全 0 矩阵)
C = np.array([
    [c1+c2, -c2,    0],
    [-c2,    c2+c3, -c3],
    [0,     -c3,    c3]
])

# 运行求解
F_input = np.array([0, 0, 0.5]) 

solve_vibration_exam(M, C, K, F_input)

1. 固有频率 (rad/s): [ 9.76061067 26.70722855 37.80938588]
2. 质量归一化振型矩阵 (每列为一个 Mode❗️):
[[ 0.57005407 -0.77075284 -1.03970112]
 [ 0.85957548 -0.36832216  0.74433877]
 [ 1.03329598  1.42361884 -0.48881711]]
------------------------------------------------------------
3. 受迫振动分析 (频率: 1.5534 Hz)
   激励力向量 F: [0.  0.  0.5] N
   质量块 m1 的位移振幅: 5.839182e-01 m
   质量块 m1 的峰峰值 (P-P): 1.167836e+00 m
   质量块 m2 的位移振幅: 8.804807e-01 m
   质量块 m2 的峰峰值 (P-P): 1.760961e+00 m
   质量块 m3 的位移振幅: 1.058441e+00 m
   质量块 m3 的峰峰值 (P-P): 2.116882e+00 m


## 激励力的峰峰值为 $1\text{ N}$。在交流电或机械振动公式中，计算通常使用幅值（Amplitude），所以你在代码输入 F_vector 时应该使用 $0.5\text{ N}$